# Code-Qualität und Refactoring

**Refactoring** bedeutet, den Aufbau von Code zu verbessern, ohne sein Verhalten zu verändern. Vergleichbar ist das mit dem Aufräumen einer Werkstatt: Danach steht dieselbe Ausstattung zur Verfügung, aber jedes Werkzeug hat seinen Platz, und die nächste Arbeit geht schneller von der Hand.

In diesem Notebook wird ein typisches "gewachsenes" Auswertungsskript Schritt für Schritt aufgeräumt. Das Skript liest Verkaufsdaten aus einer CSV-Datei, berechnet einige Kennzahlen und schreibt einen kurzen Bericht. Es funktioniert, ist aber schwer zu lesen und schwer zu ändern.

Die Theorie zu Code Smells und PEP 8 steht in den Folien zu dieser Session. Dieses Notebook zeigt die Anwendung.

⚠️ Die Zellen bauen aufeinander auf und sollten der Reihe nach ausgeführt werden. Das Notebook liest die Datei `umsaetze.csv` aus diesem Ordner und legt dort die Datei `bericht.txt` an.

## Lernziele und Inhalt

**Lernziele**

1. Code Smells wie kryptische Namen, Wiederholungen, feste Zahlenwerte und lange Codeblöcke in bestehendem Code erkennen.
2. Code in kleinen, überprüfbaren Schritten umbauen und nach jedem Schritt prüfen, dass sich das Verhalten nicht geändert hat.
3. Sprechende Namen vergeben und den Code nach PEP 8 formatieren.
4. Wiederholten Code in Funktionen auslagern und feste Werte als Konstanten benennen.
5. Komplexe Bedingungen vereinfachen und Dateien sicher mit `with` öffnen.
6. Funktionen mit Docstrings und Type Hints dokumentieren.
7. Refactoring (Verhalten bleibt gleich) von Erweiterungen und Fehlerkorrekturen (Verhalten ändert sich) unterscheiden.

**Inhalt**

- [1. Ausgangslage](#ausgangslage)
- [2. Code Smells finden](#code-smells)
- [3. Refactoring Schritt für Schritt](#schritte)
  - [Schritt 1: Sprechende Namen und PEP 8](#schritt-1)
  - [Schritt 2: Daten einmal aufbereiten](#schritt-2)
  - [Schritt 3: Wiederholungen in Funktionen auslagern](#schritt-3)
  - [Schritt 4: Feste Werte als Konstanten](#schritt-4)
  - [Schritt 5: Bedingungen vereinfachen](#schritt-5)
  - [Schritt 6: Dateien sicher öffnen](#schritt-6)
  - [Schritt 7: Docstrings und Type Hints](#schritt-7)
- [4. Nach dem Refactoring: Fehler beheben und erweitern](#erweiterung)
- [5. Zusammenfassung](#zusammenfassung)
- [Übungsaufgaben](#uebungsaufgaben)

<a id="ausgangslage"></a>

## 1. Ausgangslage

Die Datei `umsaetze.csv` enthält eine Zeile pro Bestellung. Die Spalten sind durch Semikolons getrennt, wie es bei CSV-Dateien aus einem deutschen Excel üblich ist:

In [ ]:
with open("umsaetze.csv", "r", encoding="utf-8") as csv_file:
    print(csv_file.read())

Das folgende Skript ist über die Zeit "gewachsen": Jemand hat es geschrieben, ein anderer hat eine Region ergänzt, ein Dritter die Bestseller-Auswertung. Es läuft fehlerfrei und erzeugt den gewünschten Bericht:

In [ ]:
f = open("umsaetze.csv", encoding="utf-8")
lines = f.readlines()
f.close()
d = []
for l in lines[1:]:
    d.append(l.strip().split(";"))

r = []
s = 0
for x in d:
    if x[1] == "Nord":
        s = s + int(x[3]) * float(x[4])
r.append("Umsatz Nord: " + str(round(s, 2)))
s = 0
for x in d:
    if x[1] == "Süd":
        s = s + int(x[3]) * float(x[4])
r.append("Umsatz Süd: " + str(round(s, 2)))
s2 = 0
for x in d:
    if x[1]=="West":
        s2 = s2+int(x[3])*float(x[4])
r.append("Umsatz West: " + str(round(s2, 2)))
t = 0
for x in d:
    t = t + int(x[3]) * float(x[4])
r.append("Gesamtumsatz netto: " + str(round(t, 2)))
r.append("Gesamtumsatz brutto: " + str(round(t * 1.19, 2)))
c = {}
for x in d:
    if x[2] in c:
        c[x[2]] = c[x[2]] + int(x[3])
    else:
        c[x[2]] = int(x[3])
b = ""
m = 0
for k in c:
    if c[k] > m:
        m = c[k]
        b = k
r.append("Bestseller: " + b + " (" + str(m) + " Stück)")
g = 0
for x in d:
    if int(x[3]) * float(x[4]) > 100 and not int(x[3]) * float(x[4]) <= 100 and int(x[3]) * float(x[4]) != 0:
        g = g + 1
r.append("Große Bestellungen: " + str(g))
o = open("bericht.txt", "w", encoding="utf-8")
for z in r:
    o.write(z + "\n")
o.close()
print("\n".join(r))

### Das Verhalten festhalten

Refactoring darf das Verhalten nicht verändern. Damit das nach jedem Schritt überprüfbar ist, wird das Ergebnis des Originals jetzt als **Referenz** gespeichert. Das beobachtbare Verhalten ist hier der Inhalt von `bericht.txt`.

Die Funktion `check_report()` vergleicht nach jedem Schritt den neu erzeugten Bericht mit der Referenz. Weicht er ab, löst `assert` einen `AssertionError` aus.

In [ ]:
with open("bericht.txt", "r", encoding="utf-8") as report_file:
    reference_report = report_file.read()


def check_report():
    """Compare bericht.txt with the reference report from the original script."""
    with open("bericht.txt", "r", encoding="utf-8") as report_file:
        current_report = report_file.read()
    assert current_report == reference_report, "Der Bericht hat sich verändert!"
    print("✓ Bericht unverändert")


check_report()

💡 In größeren Projekten übernehmen automatisierte Tests diese Aufgabe (Thema der Advanced Session "Unit Testing"). Das Prinzip ist dasselbe: erst das Verhalten absichern, dann umbauen.

<a id="code-smells"></a>

## 2. Code Smells finden

Bevor etwas geändert wird, lohnt sich eine Bestandsaufnahme. Im Skript finden sich unter anderem diese Code Smells:

| Code Smell | Stelle im Code | Schritt |
|---|---|---|
| Kryptische Namen | `d`, `r`, `s`, `s2`, `t`, `c`, `b`, `m`, `g`, `o`, `x`, `l`, `z` | 1 |
| Uneinheitliche Formatierung | `x[1]=="West"`, `s2+int(x[3])*float(x[4])` | 1 |
| Spaltennummern statt Namen | `x[1]`, `x[2]`, `x[3]`, `x[4]` | 2 |
| Dieselbe Umrechnung an vielen Stellen | `int(x[3]) * float(x[4])` steht siebenmal im Code | 2 |
| Wiederholter Code | drei fast identische Blöcke für die Regionen | 3 |
| Ein langer Block mit vielen Aufgaben | Einlesen, Rechnen und Schreiben ohne Struktur | 3 |
| Feste Zahlenwerte ohne Namen | `1.19`, `100` | 4 |
| Unnötig komplexe Bedingung | drei Vergleiche, die dasselbe prüfen | 5 |
| Dateien ohne `with` | `open()` und `close()` von Hand | 6 |
| Keine Dokumentation | kein Kommentar, kein Docstring | 7 |

Die Schritte im nächsten Kapitel arbeiten diese Liste nacheinander ab. Jeder Schritt ist klein und endet mit `check_report()`.

<a id="schritte"></a>

## 3. Refactoring Schritt für Schritt

<a id="schritt-1"></a>

### Schritt 1: Sprechende Namen und PEP 8

Die erste Maßnahme ändert nichts an der Logik: Jede Variable bekommt einen Namen, der sagt, was sie enthält, und die Formatierung folgt PEP 8 (Leerzeichen um Operatoren, Leerzeilen zwischen Abschnitten).

Damit `check_report()` weiterhin funktioniert, steckt der Code ab jetzt in einer Funktion `create_sales_report()`, die bei jedem Schritt neu definiert und aufgerufen wird.

In [ ]:
def create_sales_report():
    sales_file = open("umsaetze.csv", encoding="utf-8")
    lines = sales_file.readlines()
    sales_file.close()

    rows = []
    for line in lines[1:]:
        rows.append(line.strip().split(";"))

    report_lines = []

    revenue_north = 0
    for row in rows:
        if row[1] == "Nord":
            revenue_north = revenue_north + int(row[3]) * float(row[4])
    report_lines.append("Umsatz Nord: " + str(round(revenue_north, 2)))

    revenue_south = 0
    for row in rows:
        if row[1] == "Süd":
            revenue_south = revenue_south + int(row[3]) * float(row[4])
    report_lines.append("Umsatz Süd: " + str(round(revenue_south, 2)))

    revenue_west = 0
    for row in rows:
        if row[1] == "West":
            revenue_west = revenue_west + int(row[3]) * float(row[4])
    report_lines.append("Umsatz West: " + str(round(revenue_west, 2)))

    total_revenue = 0
    for row in rows:
        total_revenue = total_revenue + int(row[3]) * float(row[4])
    report_lines.append("Gesamtumsatz netto: " + str(round(total_revenue, 2)))
    report_lines.append("Gesamtumsatz brutto: " + str(round(total_revenue * 1.19, 2)))

    quantity_by_product = {}
    for row in rows:
        if row[2] in quantity_by_product:
            quantity_by_product[row[2]] = quantity_by_product[row[2]] + int(row[3])
        else:
            quantity_by_product[row[2]] = int(row[3])
    bestseller = ""
    bestseller_quantity = 0
    for product in quantity_by_product:
        if quantity_by_product[product] > bestseller_quantity:
            bestseller_quantity = quantity_by_product[product]
            bestseller = product
    report_lines.append("Bestseller: " + bestseller + " (" + str(bestseller_quantity) + " Stück)")

    large_order_count = 0
    for row in rows:
        if int(row[3]) * float(row[4]) > 100 and not int(row[3]) * float(row[4]) <= 100 and int(row[3]) * float(row[4]) != 0:
            large_order_count = large_order_count + 1
    report_lines.append("Große Bestellungen: " + str(large_order_count))

    report_file = open("bericht.txt", "w", encoding="utf-8")
    for report_line in report_lines:
        report_file.write(report_line + "\n")
    report_file.close()

In [ ]:
create_sales_report()
check_report()

Allein durch die Namen wird jetzt sichtbar, was die einzelnen Blöcke tun. Gleichzeitig fallen die Wiederholungen deutlicher auf: Die drei Regionsblöcke unterscheiden sich nur im Namen der Region.

💡 Bei `x += 1` statt `x = x + 1` handelt es sich um eine reine Stilfrage. Solche Änderungen lassen sich jederzeit nebenbei machen, sie verändern das Verhalten nicht.

<a id="schritt-2"></a>

### Schritt 2: Daten einmal aufbereiten

Jede Zeile ist bisher eine Liste von Strings. Deshalb steht überall `row[3]` statt "Menge", und an sieben Stellen wird `int(row[3]) * float(row[4])` neu berechnet.

Besser ist es, jede Zeile **einmal** beim Einlesen in ein Dictionary mit sprechenden Schlüsseln umzuwandeln und den Umsatz direkt mitzuberechnen. Mit Tuple Unpacking werden die Spalten dabei gleich benannt:

In [ ]:
def create_sales_report():
    sales_file = open("umsaetze.csv", encoding="utf-8")
    lines = sales_file.readlines()
    sales_file.close()

    sales = []
    for line in lines[1:]:
        date, region, product, quantity, unit_price = line.strip().split(";")
        sales.append({
            "date": date,
            "region": region,
            "product": product,
            "quantity": int(quantity),
            "revenue": int(quantity) * float(unit_price),
        })

    report_lines = []

    revenue_north = 0
    for sale in sales:
        if sale["region"] == "Nord":
            revenue_north += sale["revenue"]
    report_lines.append("Umsatz Nord: " + str(round(revenue_north, 2)))

    revenue_south = 0
    for sale in sales:
        if sale["region"] == "Süd":
            revenue_south += sale["revenue"]
    report_lines.append("Umsatz Süd: " + str(round(revenue_south, 2)))

    revenue_west = 0
    for sale in sales:
        if sale["region"] == "West":
            revenue_west += sale["revenue"]
    report_lines.append("Umsatz West: " + str(round(revenue_west, 2)))

    total_revenue = 0
    for sale in sales:
        total_revenue += sale["revenue"]
    report_lines.append("Gesamtumsatz netto: " + str(round(total_revenue, 2)))
    report_lines.append("Gesamtumsatz brutto: " + str(round(total_revenue * 1.19, 2)))

    quantity_by_product = {}
    for sale in sales:
        if sale["product"] in quantity_by_product:
            quantity_by_product[sale["product"]] += sale["quantity"]
        else:
            quantity_by_product[sale["product"]] = sale["quantity"]
    bestseller = ""
    bestseller_quantity = 0
    for product in quantity_by_product:
        if quantity_by_product[product] > bestseller_quantity:
            bestseller_quantity = quantity_by_product[product]
            bestseller = product
    report_lines.append("Bestseller: " + bestseller + " (" + str(bestseller_quantity) + " Stück)")

    large_order_count = 0
    for sale in sales:
        if sale["revenue"] > 100 and not sale["revenue"] <= 100 and sale["revenue"] != 0:
            large_order_count += 1
    report_lines.append("Große Bestellungen: " + str(large_order_count))

    report_file = open("bericht.txt", "w", encoding="utf-8")
    for report_line in report_lines:
        report_file.write(report_line + "\n")
    report_file.close()

In [ ]:
create_sales_report()
check_report()

⚠️ Der Umsatz wird genau wie im Original als `int(quantity) * float(unit_price)` berechnet. Eine scheinbar gleichwertige Umstellung, etwa erst alle Mengen und Preise zu summieren, könnte bei Floats durch Rundungsfehler minimal abweichende Ergebnisse liefern. Die Prüfung mit `check_report()` würde das sofort zeigen.

<a id="schritt-3"></a>

### Schritt 3: Wiederholungen in Funktionen auslagern

`create_sales_report()` erledigt noch alles auf einmal: Einlesen, mehrere Berechnungen, Text erzeugen, Datei schreiben. Jede dieser Aufgaben wird jetzt eine eigene, kleine Funktion. Die drei Regionsblöcke werden dabei zu **einer** Funktion mit der Region als Parameter, die in einer Schleife aufgerufen wird.

In [ ]:
def read_sales(path):
    sales_file = open(path, encoding="utf-8")
    lines = sales_file.readlines()
    sales_file.close()

    sales = []
    for line in lines[1:]:
        date, region, product, quantity, unit_price = line.strip().split(";")
        sales.append({
            "date": date,
            "region": region,
            "product": product,
            "quantity": int(quantity),
            "revenue": int(quantity) * float(unit_price),
        })
    return sales


def calculate_region_revenue(sales, region):
    revenue = 0
    for sale in sales:
        if sale["region"] == region:
            revenue += sale["revenue"]
    return revenue


def calculate_total_revenue(sales):
    total_revenue = 0
    for sale in sales:
        total_revenue += sale["revenue"]
    return total_revenue


def find_bestseller(sales):
    quantity_by_product = {}
    for sale in sales:
        if sale["product"] in quantity_by_product:
            quantity_by_product[sale["product"]] += sale["quantity"]
        else:
            quantity_by_product[sale["product"]] = sale["quantity"]
    bestseller = ""
    bestseller_quantity = 0
    for product in quantity_by_product:
        if quantity_by_product[product] > bestseller_quantity:
            bestseller_quantity = quantity_by_product[product]
            bestseller = product
    return bestseller, bestseller_quantity


def count_large_orders(sales):
    large_order_count = 0
    for sale in sales:
        if sale["revenue"] > 100 and not sale["revenue"] <= 100 and sale["revenue"] != 0:
            large_order_count += 1
    return large_order_count


def create_report_lines(sales):
    report_lines = []
    for region in ["Nord", "Süd", "West"]:
        revenue = calculate_region_revenue(sales, region)
        report_lines.append(f"Umsatz {region}: {round(revenue, 2)}")

    total_revenue = calculate_total_revenue(sales)
    report_lines.append(f"Gesamtumsatz netto: {round(total_revenue, 2)}")
    report_lines.append(f"Gesamtumsatz brutto: {round(total_revenue * 1.19, 2)}")

    bestseller, bestseller_quantity = find_bestseller(sales)
    report_lines.append(f"Bestseller: {bestseller} ({bestseller_quantity} Stück)")

    report_lines.append(f"Große Bestellungen: {count_large_orders(sales)}")
    return report_lines


def write_report(report_lines, path):
    report_file = open(path, "w", encoding="utf-8")
    for report_line in report_lines:
        report_file.write(report_line + "\n")
    report_file.close()


def create_sales_report():
    sales = read_sales("umsaetze.csv")
    report_lines = create_report_lines(sales)
    write_report(report_lines, "bericht.txt")

In [ ]:
create_sales_report()
check_report()

`create_sales_report()` liest sich jetzt wie eine Inhaltsangabe: Daten einlesen, Berichtszeilen erzeugen, Bericht schreiben. Jede Teilaufgabe lässt sich einzeln verstehen, testen und ändern.

Die Textbausteine werden jetzt mit f-Strings erzeugt. `f"{round(revenue, 2)}"` ergibt dabei genau denselben Text wie `str(round(revenue, 2))`.

⚠️ Eine schönere Formatierung wie `f"{revenue:.2f}"` wäre an dieser Stelle **kein** Refactoring mehr: Aus `111.7` würde `111.70`, der Bericht sähe anders aus. Solche Änderungen am Verhalten gehören in einen eigenen, bewussten Schritt (siehe [Kapitel 4](#erweiterung)).

💡 Ab hier wird bei jedem Schritt nur noch die Funktion neu definiert, die sich ändert. Python verwendet beim nächsten Aufruf von `create_sales_report()` automatisch die neueste Fassung.

<a id="schritt-4"></a>

### Schritt 4: Feste Werte als Konstanten

Im Code stehen noch Werte, deren Bedeutung man erraten muss oder die bei einer Änderung an mehreren Stellen angepasst werden müssten: der Mehrwertsteuersatz `1.19`, die Grenze für große Bestellungen `100`, die Liste der Regionen und die Dateinamen.

Solche Werte werden als **Konstanten** am Anfang definiert. Nach PEP 8 werden Konstanten in `UPPER_CASE` geschrieben.

In [ ]:
SALES_FILE = "umsaetze.csv"
REPORT_FILE = "bericht.txt"
REGIONS = ["Nord", "Süd", "West"]
VAT_RATE = 0.19
LARGE_ORDER_THRESHOLD = 100


def count_large_orders(sales):
    large_order_count = 0
    for sale in sales:
        if sale["revenue"] > LARGE_ORDER_THRESHOLD and not sale["revenue"] <= LARGE_ORDER_THRESHOLD and sale["revenue"] != 0:
            large_order_count += 1
    return large_order_count


def create_report_lines(sales):
    report_lines = []
    for region in REGIONS:
        revenue = calculate_region_revenue(sales, region)
        report_lines.append(f"Umsatz {region}: {round(revenue, 2)}")

    total_revenue = calculate_total_revenue(sales)
    report_lines.append(f"Gesamtumsatz netto: {round(total_revenue, 2)}")
    report_lines.append(f"Gesamtumsatz brutto: {round(total_revenue * (1 + VAT_RATE), 2)}")

    bestseller, bestseller_quantity = find_bestseller(sales)
    report_lines.append(f"Bestseller: {bestseller} ({bestseller_quantity} Stück)")

    report_lines.append(f"Große Bestellungen: {count_large_orders(sales)}")
    return report_lines


def create_sales_report():
    sales = read_sales(SALES_FILE)
    report_lines = create_report_lines(sales)
    write_report(report_lines, REPORT_FILE)

In [ ]:
create_sales_report()
check_report()

💡 `total_revenue * (1 + VAT_RATE)` liefert hier dasselbe Ergebnis wie `total_revenue * 1.19`, da `1 + 0.19` exakt `1.19` ergibt. `check_report()` bestätigt das.

<a id="schritt-5"></a>

### Schritt 5: Bedingungen vereinfachen

Die Bedingung in `count_large_orders()` prüft dreimal dasselbe: Ist ein Wert größer als 100, ist er automatisch nicht kleiner oder gleich 100 und auch nicht 0. Übrig bleibt eine einzige Bedingung.

Gleichzeitig lassen sich die Schleifen, die nur zählen oder summieren, mit Comprehensions und den eingebauten Funktionen `len()` und `sum()` kürzer schreiben. Auch die Bestseller-Suche wird einfacher: `.get()` mit Standardwert ersetzt das `if`-`else`, und `max()` mit `key` findet den größten Wert direkt.

In [ ]:
def count_large_orders(sales):
    return len([sale for sale in sales if sale["revenue"] > LARGE_ORDER_THRESHOLD])


def calculate_region_revenue(sales, region):
    return sum([sale["revenue"] for sale in sales if sale["region"] == region])


def calculate_total_revenue(sales):
    return sum([sale["revenue"] for sale in sales])


def find_bestseller(sales):
    quantity_by_product = {}
    for sale in sales:
        product = sale["product"]
        quantity_by_product[product] = quantity_by_product.get(product, 0) + sale["quantity"]
    if not quantity_by_product:
        return "", 0
    bestseller = max(quantity_by_product, key=quantity_by_product.get)
    return bestseller, quantity_by_product[bestseller]

In [ ]:
create_sales_report()
check_report()

💡 `max(quantity_by_product, key=quantity_by_product.get)` durchläuft die Schlüssel des Dictionaries und vergleicht sie anhand des zugehörigen Werts, genau wie `key` bei `sorted()` aus der Session "Funktionen". Bei Gleichstand liefert `max()` den ersten Treffer, genau wie die ursprüngliche Schleife.

⚠️ Die Zeile `if not quantity_by_product: return "", 0` ist nötig, damit das Verhalten auch bei leeren Daten gleich bleibt: Die ursprüngliche Schleife lieferte dann `("", 0)`, `max()` auf einem leeren Dictionary würde dagegen einen `ValueError` auslösen. Beim Refactoring lohnt sich immer der Blick auf solche Randfälle.

<a id="schritt-6"></a>

### Schritt 6: Dateien sicher öffnen

Beide Dateizugriffe verwenden noch `open()` und `close()` von Hand. Tritt dazwischen ein Fehler auf, bleibt die Datei offen. Mit `with` wird sie in jedem Fall geschlossen (Session "Dateien").

In [ ]:
def read_sales(path):
    sales = []
    with open(path, "r", encoding="utf-8") as sales_file:
        sales_file.readline()   # Kopfzeile überspringen
        for line in sales_file:
            date, region, product, quantity, unit_price = line.strip().split(";")
            sales.append({
                "date": date,
                "region": region,
                "product": product,
                "quantity": int(quantity),
                "revenue": int(quantity) * float(unit_price),
            })
    return sales


def write_report(report_lines, path):
    with open(path, "w", encoding="utf-8") as report_file:
        for report_line in report_lines:
            report_file.write(report_line + "\n")

In [ ]:
create_sales_report()
check_report()

<a id="schritt-7"></a>

### Schritt 7: Docstrings und Type Hints

Zum Schluss bekommt jede Funktion einen Docstring, der ihren Zweck beschreibt, und Type Hints für Parameter und Rückgabewert (Session "Erweiterte Python-Syntax"). Die folgende Zelle zeigt den vollständigen Endstand an einem Stück.

Ein Datensatz ist hier ein Dictionary mit gemischten Werttypen. Der Type Hint `dict[str, str | int | float]` beschreibt das genau, ist aber lang. Deshalb wird er einmal unter dem Namen `Sale` gespeichert und dann überall verwendet.

In [ ]:
SALES_FILE = "umsaetze.csv"
REPORT_FILE = "bericht.txt"
REGIONS = ["Nord", "Süd", "West"]
VAT_RATE = 0.19
LARGE_ORDER_THRESHOLD = 100

Sale = dict[str, str | int | float]


def read_sales(path: str) -> list[Sale]:
    """Read sales from a semicolon-separated file and compute the revenue per sale."""
    sales = []
    with open(path, "r", encoding="utf-8") as sales_file:
        sales_file.readline()   # Kopfzeile überspringen
        for line in sales_file:
            date, region, product, quantity, unit_price = line.strip().split(";")
            sales.append({
                "date": date,
                "region": region,
                "product": product,
                "quantity": int(quantity),
                "revenue": int(quantity) * float(unit_price),
            })
    return sales


def calculate_region_revenue(sales: list[Sale], region: str) -> float:
    """Return the total revenue of all sales in the given region."""
    return sum([sale["revenue"] for sale in sales if sale["region"] == region])


def calculate_total_revenue(sales: list[Sale]) -> float:
    """Return the total revenue of all sales."""
    return sum([sale["revenue"] for sale in sales])


def find_bestseller(sales: list[Sale]) -> tuple[str, int]:
    """Return the product with the highest total quantity and that quantity."""
    quantity_by_product = {}
    for sale in sales:
        product = sale["product"]
        quantity_by_product[product] = quantity_by_product.get(product, 0) + sale["quantity"]
    if not quantity_by_product:
        return "", 0
    bestseller = max(quantity_by_product, key=quantity_by_product.get)
    return bestseller, quantity_by_product[bestseller]


def count_large_orders(sales: list[Sale]) -> int:
    """Return the number of sales with a revenue above LARGE_ORDER_THRESHOLD."""
    return len([sale for sale in sales if sale["revenue"] > LARGE_ORDER_THRESHOLD])


def create_report_lines(sales: list[Sale]) -> list[str]:
    """Create the lines of the sales report."""
    report_lines = []
    for region in REGIONS:
        revenue = calculate_region_revenue(sales, region)
        report_lines.append(f"Umsatz {region}: {round(revenue, 2)}")

    total_revenue = calculate_total_revenue(sales)
    report_lines.append(f"Gesamtumsatz netto: {round(total_revenue, 2)}")
    report_lines.append(f"Gesamtumsatz brutto: {round(total_revenue * (1 + VAT_RATE), 2)}")

    bestseller, bestseller_quantity = find_bestseller(sales)
    report_lines.append(f"Bestseller: {bestseller} ({bestseller_quantity} Stück)")

    report_lines.append(f"Große Bestellungen: {count_large_orders(sales)}")
    return report_lines


def write_report(report_lines: list[str], path: str) -> None:
    """Write the report lines to a text file, one line per entry."""
    with open(path, "w", encoding="utf-8") as report_file:
        for report_line in report_lines:
            report_file.write(report_line + "\n")


def create_sales_report() -> None:
    """Read the sales data, create the report and write it to REPORT_FILE."""
    sales = read_sales(SALES_FILE)
    report_lines = create_report_lines(sales)
    write_report(report_lines, REPORT_FILE)

In [ ]:
create_sales_report()
check_report()

### Vorher und nachher

Der Endstand ist länger als das Original. Das ist kein Widerspruch: Ziel des Refactorings war nicht weniger Code, sondern Code, der sich schneller verstehen und sicherer ändern lässt.

| | Vorher | Nachher |
|---|---|---|
| Namen | `d`, `s2`, `x[3]` | `sales`, `calculate_region_revenue`, `sale["quantity"]` |
| Struktur | ein Block für alles | sieben Funktionen mit je einer Aufgabe |
| Neue Region ergänzen | Block kopieren und anpassen | Eintrag in `REGIONS` ergänzen |
| Mehrwertsteuer ändern | Zahl im Code suchen | `VAT_RATE` anpassen |
| Dateien | `open()`/`close()` von Hand | `with` |
| Dokumentation | keine | Docstrings und Type Hints |

<a id="erweiterung"></a>

## 4. Nach dem Refactoring: Fehler beheben und erweitern

Beim Aufräumen ist etwas aufgefallen, das vorher in den kopierten Blöcken unterging: Die Daten enthalten auch die Region **Ost**, der Bericht zeigt sie aber nicht. Die Summe der drei Regionen ergibt deshalb nicht den Gesamtumsatz.

Diesen Fehler zu beheben, ist **kein** Refactoring, denn der Bericht ändert sich. Deshalb geschieht das erst jetzt, als eigener, bewusster Schritt, und die Referenz wird anschließend absichtlich aktualisiert.

Dank des Refactorings sind dafür nur kleine Änderungen nötig:

- Die Regionen werden aus den Daten ermittelt, statt sie fest vorzugeben. Neue Regionen erscheinen damit automatisch.
- Die Beträge werden einheitlich mit zwei Nachkommastellen ausgegeben (`111.70` statt `111.7`).
- Eine fehlende Eingabedatei führt zu einer verständlichen Meldung statt zu einem Traceback.

In [ ]:
def get_regions(sales: list[Sale]) -> list[str]:
    """Return all regions in the sales data in order of first occurrence."""
    regions = []
    for sale in sales:
        if sale["region"] not in regions:
            regions.append(sale["region"])
    return regions


def create_report_lines(sales: list[Sale]) -> list[str]:
    """Create the lines of the sales report."""
    report_lines = []
    for region in get_regions(sales):
        revenue = calculate_region_revenue(sales, region)
        report_lines.append(f"Umsatz {region}: {revenue:.2f}")

    total_revenue = calculate_total_revenue(sales)
    report_lines.append(f"Gesamtumsatz netto: {total_revenue:.2f}")
    report_lines.append(f"Gesamtumsatz brutto: {total_revenue * (1 + VAT_RATE):.2f}")

    bestseller, bestseller_quantity = find_bestseller(sales)
    report_lines.append(f"Bestseller: {bestseller} ({bestseller_quantity} Stück)")

    report_lines.append(f"Große Bestellungen: {count_large_orders(sales)}")
    return report_lines


def create_sales_report() -> None:
    """Read the sales data, create the report and write it to REPORT_FILE."""
    try:
        sales = read_sales(SALES_FILE)
    except FileNotFoundError:
        print(f"Die Datei {SALES_FILE} wurde nicht gefunden.")
        return
    report_lines = create_report_lines(sales)
    write_report(report_lines, REPORT_FILE)


create_sales_report()

with open(REPORT_FILE, "r", encoding="utf-8") as report_file:
    print(report_file.read())

Jetzt würde `check_report()` absichtlich fehlschlagen, denn der Bericht hat sich gewollt verändert. Nach einer bewussten Änderung wird die Referenz deshalb aktualisiert, damit sie für zukünftiges Refactoring wieder als Maßstab dient:

In [ ]:
with open(REPORT_FILE, "r", encoding="utf-8") as report_file:
    reference_report = report_file.read()

check_report()

💡 Diese Trennung ist ein zentraler Grundsatz: Entweder wird umgebaut (Verhalten bleibt gleich) **oder** das Verhalten wird geändert, aber nie beides im selben Schritt. So ist bei einer Abweichung immer klar, ob sie gewollt ist.

<a id="zusammenfassung"></a>

## 5. Zusammenfassung

Refactoring verbessert die Struktur von Code, ohne sein Verhalten zu verändern. Damit das gelingt, wird das Verhalten vorher festgehalten und nach jedem kleinen Schritt überprüft.

### Behandelte Konzepte

1. Vor dem Umbau eine Referenz für das Verhalten festhalten und nach jedem Schritt damit vergleichen.
2. Code Smells systematisch erfassen und nacheinander abarbeiten.
3. Kryptische Namen durch sprechende ersetzen und nach PEP 8 formatieren.
4. Daten beim Einlesen einmal in eine gut benannte Struktur umwandeln, statt mit Spaltennummern zu arbeiten.
5. Wiederholten Code und lange Blöcke in Funktionen mit je einer Aufgabe aufteilen.
6. Feste Werte als Konstanten in `UPPER_CASE` benennen.
7. Redundante Bedingungen vereinfachen, ohne Randfälle wie leere Daten zu übersehen.
8. Funktionen mit Docstrings und Type Hints dokumentieren.
9. Refactoring und Verhaltensänderungen in getrennten Schritten durchführen.

### Checkliste für eigenen Code

| Frage | Mögliche Maßnahme |
|---|---|
| Versteht man ohne Erklärung, was eine Variable enthält? | sprechenden Namen vergeben |
| Steht derselbe Code mehrfach da? | in eine Funktion mit Parametern auslagern |
| Tut ein Codeblock mehrere Dinge? | in mehrere Funktionen aufteilen |
| Gibt es Zahlen, deren Bedeutung man erraten muss? | als Konstante benennen |
| Ist eine Bedingung schwer zu lesen? | vereinfachen oder in eine Variable mit sprechendem Namen auslagern |
| Werden Dateien ohne `with` geöffnet? | auf `with` umstellen |
| Fehlt eine Beschreibung, was eine Funktion tut? | Docstring und Type Hints ergänzen |

### Ausblick

In den Advanced Sessions geht es unter anderem um automatisierte Tests mit `pytest`. Sie übernehmen die Rolle von `check_report()` und machen Refactoring auch in großen Projekten sicher.

<a id="uebungsaufgaben"></a>

## Übungsaufgaben

Jede Aufgabe enthält eine Funktion mit Code Smells. Schreibe darunter eine verbesserte Fassung mit **gleichem Verhalten**. Die vorbereiteten Prüfzeilen vergleichen alte und neue Fassung für mehrere Eingaben und melden mit einem `AssertionError`, wenn sich das Verhalten unterscheidet.

Die Aufgaben sind unabhängig vom Sales-Beispiel oben und lassen sich direkt ausführen.

⚠️ Die Zellen brechen vor dem Lösen mit einem `NameError` ab, weil die neue Fassung der Funktion noch fehlt.

#### Aufgabe 1: Sprechende Namen

Die Funktion gibt die Namen aller Produkte zurück, deren Preis über einer Grenze liegt. Vergib sprechende Namen für die Funktion, die Parameter und alle Variablen.

In [ ]:
def calc(l, p):
    r = []
    for i in l:
        if i[1] > p:
            r.append(i[0])
    return r


# TODO: verbesserte Fassung unter dem Namen filter_expensive_products


products = [("Kaffee", 4.99), ("Olivenöl", 12.5), ("Tee", 3.2), ("Honig", 10.0)]
for limit in [0, 4, 10, 20]:
    assert filter_expensive_products(products, limit) == calc(products, limit)
print("✓ Verhalten unverändert")

#### Aufgabe 2: Wiederholungen entfernen

Die Funktion berechnet die Versandkosten für drei Länder. Die Berechnung folgt jedes Mal demselben Schema: Grundgebühr plus Gebühr pro Kilogramm, höchstens aber ein Maximalbetrag. Fasse das Schema in einer Hilfsfunktion zusammen und verwende sie für alle drei Länder.

In [ ]:
def shipping(c, w):
    if c == "DE":
        x = 4.99 + w * 0.5
        if x > 15:
            x = 15
        return x
    if c == "AT":
        x = 7.99 + w * 0.8
        if x > 25:
            x = 25
        return x
    if c == "CH":
        x = 12.99 + w * 1.2
        if x > 40:
            x = 40
        return x
    return None


# TODO: Hilfsfunktion calculate_shipping_cost(weight, base_fee, fee_per_kg, max_cost)
# TODO: verbesserte Fassung unter dem Namen get_shipping_cost(country, weight)


for country in ["DE", "AT", "CH", "FR"]:
    for weight in [0, 1.5, 10, 50]:
        assert get_shipping_cost(country, weight) == shipping(country, weight)
print("✓ Verhalten unverändert")

#### Aufgabe 3: Feste Werte als Konstanten

Die Funktion berechnet den Endpreis einer Bestellung. Ersetze alle festen Zahlenwerte durch Konstanten mit sprechenden Namen.

Tipp: Lebensmittel haben einen ermäßigten Mehrwertsteuersatz von 7 %, alle anderen Produkte 19 %. Ab einem Warenwert von 50 Euro entfallen die Versandkosten von 4.95 Euro.

In [ ]:
def final_price(net, food):
    if food:
        gross = net * 1.07
    else:
        gross = net * 1.19
    if net < 50:
        gross = gross + 4.95
    return round(gross, 2)


# TODO: Konstanten definieren
# TODO: verbesserte Fassung unter dem Namen calculate_final_price(net_amount, is_food)


for net_amount in [10, 49.99, 50, 120]:
    for is_food in [True, False]:
        assert calculate_final_price(net_amount, is_food) == final_price(net_amount, is_food)
print("✓ Verhalten unverändert")

#### Aufgabe 4: Verschachtelte Bedingungen vereinfachen

Die Funktion prüft, ob jemand ein Auto mieten darf. Ersetze die verschachtelten `if`-Anweisungen durch eine einzige, gut lesbare Bedingung.

Tipp: `has_license == True` lässt sich kürzer als `has_license` schreiben, `has_accidents == False` als `not has_accidents`.

In [ ]:
def check(age, has_license, years, has_accidents):
    if has_license == True:
        if age >= 21:
            if years >= 2:
                if has_accidents == False:
                    return True
                else:
                    return False
            else:
                return False
        else:
            return False
    else:
        return False


# TODO: verbesserte Fassung unter dem Namen can_rent_car(age, has_license, years_of_driving, has_accidents)


for age in [18, 21, 40]:
    for has_license in [True, False]:
        for years in [0, 2, 10]:
            for has_accidents in [True, False]:
                assert can_rent_car(age, has_license, years, has_accidents) == check(age, has_license, years, has_accidents)
print("✓ Verhalten unverändert")

#### Aufgabe 5: PEP 8

Die Funktion berechnet den Body-Mass-Index und ordnet ihn ein. Sie funktioniert, verstößt aber gegen zahlreiche PEP-8-Regeln. Formatiere sie neu und vergib dabei auch sprechende Namen.

Achte auf: Funktionsnamen in `snake_case`, Leerzeichen um Operatoren und nach Kommas, eine Anweisung pro Zeile, Einrückung mit 4 Leerzeichen.

In [ ]:
def BMI( W,H ):
  b=W/(H*H)
  if b<18.5:return "Untergewicht"
  elif b<25 :return "Normalgewicht"
  else:
        return "Übergewicht"


# TODO: verbesserte Fassung unter dem Namen classify_bmi(weight_kg, height_m)


for weight_kg in [50, 70, 95]:
    for height_m in [1.60, 1.75, 1.90]:
        assert classify_bmi(weight_kg, height_m) == BMI(weight_kg, height_m)
print("✓ Verhalten unverändert")

#### Aufgabe 6: Lange Funktion aufteilen

Die Funktion erledigt drei Aufgaben auf einmal: einen Text mit Noten zerlegen, Kennzahlen berechnen und einen Bericht formatieren. Teile sie in drei Funktionen auf:

- `parse_grades(text)`: gibt ein Dictionary `{Name: Note}` zurück
- `calculate_statistics(grades)`: gibt Durchschnitt und Name der besten Person zurück
- `format_grade_report(text)`: nutzt die beiden anderen Funktionen und gibt den fertigen Bericht zurück

Verwende dabei gern Comprehensions und Tuple Unpacking.

In [ ]:
def report(t):
    g = {}
    for p in t.split(","):
        n = p.split(":")[0].strip()
        v = int(p.split(":")[1].strip())
        g[n] = v
    s = 0
    for n in g:
        s = s + g[n]
    a = s / len(g)
    b = ""
    bv = 7
    for n in g:
        if g[n] < bv:
            bv = g[n]
            b = n
    return "Teilnehmende: " + str(len(g)) + "\nDurchschnitt: " + str(round(a, 2)) + "\nBeste Note: " + b


# TODO: parse_grades(text)
# TODO: calculate_statistics(grades)
# TODO: format_grade_report(text)


for grade_text in ["Anna: 2, Bernd: 1, Claudia: 3", "Dora: 4", "Emil: 2, Fatma: 2, Gerd: 5"]:
    assert format_grade_report(grade_text) == report(grade_text)
print(format_grade_report("Anna: 2, Bernd: 1, Claudia: 3"))
print("✓ Verhalten unverändert")

#### Aufgabe 7: Den Nutzen zeigen

Erweitere den aufgeräumten Sales-Bericht aus diesem Notebook um eine Zeile, die den durchschnittlichen Umsatz pro Bestellung ausgibt, zum Beispiel `Durchschnitt pro Bestellung: 64.25`.

Das ist eine bewusste Verhaltensänderung, keine Refactoring-Aufgabe. Überlege anschließend: Wie viele Stellen hättest du im ursprünglichen Skript anpassen müssen, und wie viele waren es jetzt?

Tipp: Die Funktionen aus Kapitel 3 und 4 stehen weiterhin zur Verfügung, wenn das Notebook vorher von oben nach unten ausgeführt wurde.

In [ ]:
# TODO: Funktion calculate_average_revenue(sales) schreiben
# TODO: create_report_lines(sales) um die neue Zeile erweitern
# TODO: create_sales_report() aufrufen und bericht.txt ausgeben